In [1]:
#!/usr/bin/env python3
import os
import sys
import glob
import re
import numpy as np
import pandas as pd
import h5py
import matplotlib
import matplotlib.pyplot as plt
import logging

import torch
import tensorflow as tf
import keras
from bm3d import bm3d

# SCUNet imports
module_dir_scu = "/global/u2/k/kberard/SCGSR/Research/Diamond/stock_models/SCUNet" 
sys.path.insert(0, module_dir_scu)
from models.network_scunet import SCUNet as SCUNet

# FFT imports
qmc_algo_path = os.path.abspath('/pscratch/sd/k/kberard/SCGSR/3D_VMC/Model_Train_dat/FFT_Jaron/qmc_algo_tools') 
dev = os.path.abspath('/pscratch/sd/k/kberard/SCGSR/3D_VMC/Model_Train_dat/FFT_Jaron/developer_tools') 
sys.path.insert(0, qmc_algo_path) 
sys.path.insert(0, dev) 
from qmc_algo_tools.density_denoise import DensityFourierFilterErrorCeil

# ==========================================
# 0. CONFIGURATION & PATCHES
# ==========================================
model_samp = str(240700000)
base_dir = "/pscratch/sd/k/kberard/SCGSR/Data/diamond_1x1x1_bfd/density_data/vmc_J2"
ref_path = os.path.join(base_dir, "density_tot_ref_mean.h5")
dft_path = '/global/u2/k/kberard/SCGSR/Research/Diamond/Data/density_tot_ref.h5'

# CPU-loading & Unpickling patch for torch to handle PyTorch 2.6+ backward compatibility safely
_original_torch_load = torch.load
def torch_load_cpu(*args, **kwargs):
    if 'map_location' not in kwargs:
        kwargs['map_location'] = torch.device('cpu')
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False  # Allows unpickling full structural model containers
    return _original_torch_load(*args, **kwargs)
torch.load = torch_load_cpu

keras.config.enable_unsafe_deserialization()
torch.serialization.add_safe_globals([SCUNet])

# ==========================================
# 1. UNIFIED MATH & UTILITY FUNCTIONS
# ==========================================
def D_JS(p1, p2, tol=1e-16):
    """Calculates Jensen-Shannon Divergence."""
    p1 = p1 / (np.sum(p1) + 1e-16)
    p2 = p2 / (np.sum(p2) + 1e-16)
    pm = (p1 + p2) / 2
    p1_nonzero, p2_nonzero = np.abs(p1) > tol, np.abs(p2) > tol
    
    d = 0.5 * (
        (np.abs(p1[p1_nonzero]) * np.log(np.abs(p1[p1_nonzero]) / np.abs(pm[p1_nonzero]))).sum() + 
        (np.abs(p2[p2_nonzero]) * np.log(np.abs(p2[p2_nonzero]) / np.abs(pm[p2_nonzero]))).sum()
    )
    return d / np.log(2)

# Transform density
def transform(density, density_ref, transform_type):
    if transform_type == 'value':
        return density
    elif transform_type == 'sqrt':
        return np.sqrt(np.abs(density))
    elif transform_type == 'log':
        return np.log(np.abs(density))
    elif transform_type == 'residual_noise':
        return (density - density_ref) / (np.sqrt(np.abs(density_ref)) + 1e-12) # Added tiny epsilon for safety
    else:
        raise RuntimeError('bad transform type')

def inverse_transform(density_trans, density_ref, transform_type):
    if transform_type == 'value':
        return density_trans
    elif transform_type == 'sqrt':
        return density_trans**2
    elif transform_type == 'log':
        return np.exp(density_trans) # NOTE: inverse of log is exp, not log(abs(dtrans))
    elif transform_type == 'residual_noise':
        return density_ref + (np.sqrt(np.abs(density_ref)) * density_trans)
    else:
        raise RuntimeError('bad transform type')

def encode_voxel_to_rgb_global(vol_3d):
    """Normalizes the ENTIRE 3D volume, preventing slice-by-slice distortion."""
    v_min, v_max = float(vol_3d.min()), float(vol_3d.max())
    if v_max == v_min: v_max = v_min + 1e-6
    
    normed = (vol_3d - v_min) / (v_max - v_min)
    rgb_volume = np.stack([normed]*3, axis=-1).astype(np.float32)
    return rgb_volume, v_min, v_max

def decode_rgb_to_voxel_global(rgb_volume, v_min, v_max):
    """Restores the 3D volume using the global scalars."""
    gray = rgb_volume[:, :, :, 0]
    return gray * (v_max - v_min) + v_min

# ==========================================
# 2. KERAS CUSTOM OBJECTS
# ==========================================
@tf.keras.utils.register_keras_serializable()
class OnesLikeLayer(tf.keras.layers.Layer):
    def call(self, inputs): return tf.ones_like(inputs)

def ones_like_fn(a): return tf.ones_like(a)

@tf.keras.utils.register_keras_serializable()
class RenormalizeToEight(tf.keras.layers.Layer):
    def call(self, x):
        total = tf.reduce_sum(x, axis=[1, 2, 3, 4], keepdims=True)
        return x / (total + 1e-8) * 8.0

@tf.keras.utils.register_keras_serializable()
def jensen_shannon_divergence_loss(y_true, y_pred):
    return 0.0 # Dummy function just for loading the model structure safely

# ==========================================
# 3. UNIFIED INFERENCE DISPATCHER
# ==========================================
def denoise_with_scunet(rgb_image_np, model, device):
    img = np.clip(rgb_image_np.astype(np.float32), 0, 1)
    img_tensor = torch.from_numpy(np.transpose(img, (2, 0, 1))).float().unsqueeze(0).to(device)
    with torch.no_grad():
        output_tensor = model(img_tensor)
    output_np = output_tensor.squeeze().cpu().detach().numpy()
    if output_np.ndim == 3:
        output_np = np.transpose(output_np, (1, 2, 0))
    return np.clip(output_np, 0, 1)

def run_model_inference(test_d, ref_d_dft, model_name, models_dict, transform_type='residual_noise'):
    """Routes the input data to the appropriate model logic."""
    
    # 1. ALL METHODS: Transform data into target residual/representation space
    trans_d = transform(test_d, ref_d_dft, transform_type)

    if model_name in ['scunet_pre', 'scunet_trained', 'scunet_ft', 'CAE', 'nature', 'bm3d']:
        rgb_vol, v_min, v_max = encode_voxel_to_rgb_global(trans_d)
        denoised_rgb = np.zeros_like(rgb_vol)

        if model_name == 'bm3d':
            sigma = np.sqrt(1.0 / 100)
            for i in range(trans_d.shape[0]):
                denoised_gray = bm3d(rgb_vol[i, :, :, 0], sigma_psd=sigma)
                denoised_rgb[i] = np.stack([denoised_gray]*3, axis=-1)

        elif model_name.startswith('scunet'):
            model = models_dict[model_name]
            device = 'cuda' if torch.cuda.is_available() else 'cpu'
            model.to(device)
            for i in range(trans_d.shape[0]):
                denoised_rgb[i] = denoise_with_scunet(rgb_vol[i], model, device)

        elif model_name == 'CAE':
            denoised_rgb = models_dict['CAE'].predict(rgb_vol, verbose=0)
            
        elif model_name == 'nature':
            denoised_rgb = models_dict['nature'].predict(rgb_vol, verbose=0)

        # Restore from RGB normalization
        denoised_trans = decode_rgb_to_voxel_global(denoised_rgb, v_min, v_max)

    elif model_name == 'CAE_3D':
        # Add channel/batch dims strictly for Keras
        reshaped_arr = trans_d.reshape(1, 64, 64, 64, 1)
        denoised_raw = models_dict['CAE_3D'].predict(reshaped_arr, verbose=0)
        denoised_trans = denoised_raw.reshape(64, 64, 64)
        
    elif model_name == 'FFT':
        broken!
        dm = DensityFourierFilterErrorCeil(density_ref=ref_d_dft, filter_mode='augment')
        denoised_trans = dm.denoise(trans_d)

    else:
        raise ValueError(f"Unsupported model: {model_name}")

    # 2. ALL METHODS: Inverse transform back to physical density
    return inverse_transform(denoised_trans, ref_d_dft, transform_type)

def evaluate_and_enforce(denoised_d, ref_d):
    """Enforces non-negativity and strictly normalizes to 8 electrons before scoring."""
    denoised_d = np.maximum(denoised_d, 0.0)
    denoised_d = denoised_d * (8.0 / (np.sum(denoised_d) + 1e-16))
    return D_JS(denoised_d, ref_d)

# ==========================================
# 4. MAIN EXECUTION PIPELINE
# ==========================================
def main():
    print("Loading Base Data & DFT Reference...")
    with h5py.File(ref_path, 'r') as file:
        ref_d_mean = file['density'][:]
        ref_d_mean = ref_d_mean * (8.0 / np.sum(ref_d_mean))

    with h5py.File(dft_path, 'r') as file:
        dft_d = file['density'][:]
        dft_d = dft_d * (8.0 / np.sum(dft_d))
        
    DFT_vs_VMC = D_JS(ref_d_mean, dft_d)
    noisy_files = sorted(glob.glob(os.path.join(base_dir, "density_tot_vmc_mean*.h5")))
    print(f"Found {len(noisy_files)} noisy files.")

    # All models to evaluate
    models_to_run = ['scunet_trained', 'scunet_pre', 'scunet_ft', 'nature', 'CAE', 'CAE_3D', 'bm3d', 'FFT']
    
    print("Pre-loading Models into Memory...")
    models_dict = {}
    
    # Load PyTorch Models
    if 'scunet_pre' in models_to_run:
        m = SCUNet(in_nc=3, config=[4, 4, 4, 4, 4, 4, 4], dim=64)
        m.load_state_dict(torch.load('/global/u2/k/kberard/SCGSR/Research/Diamond/stock_models/SCUNet/model_zoo/scunet_color_25.pth', map_location='cpu'))
        m.eval()
        models_dict['scunet_pre'] = m
    if 'scunet_trained' in models_to_run:
        models_dict['scunet_trained'] = torch.load(f'/pscratch/sd/k/kberard/SCGSR/EDDA/Diamond/Image_Models_NO_DFT/Scunet_trained_Models/10000000scunet_trained', map_location='cpu').eval()
    if 'scunet_ft' in models_to_run:
        models_dict['scunet_ft'] = torch.load(f'/pscratch/sd/k/kberard/SCGSR/EDDA/Diamond/Image_Models_NO_DFT/Scunet_FT_Models/10000000scunet_FT', map_location='cpu').eval()

    # Load Keras Models
    if 'CAE' in models_to_run:
        models_dict['CAE'] = tf.keras.models.load_model(f'/pscratch/sd/k/kberard/SCGSR/EDDA/Diamond/Image_Models_NO_DFT/CAE_img_Models/CAE_IMG_enc.keras')
    if 'nature' in models_to_run:
        models_dict['nature'] = tf.keras.models.load_model(f'/pscratch/sd/k/kberard/SCGSR/3D_VMC/Model_Train_dat/Nature_Models/{model_samp}_Nature.keras', custom_objects={'ones_like_fn': ones_like_fn})
    if 'CAE_3D' in models_to_run:
        models_dict['CAE_3D'] = tf.keras.models.load_model(f'/pscratch/sd/k/kberard/SCGSR/3D_VMC/Model_Train_dat/CAE_3D_Models/{model_samp}CAE_3D.keras', custom_objects={"RenormalizeToEight": RenormalizeToEight, "jensen_shannon_divergence_loss": jensen_shannon_divergence_loss})

    # Prepare results storage
    results_dict = {model: [] for model in models_to_run}
    results_ref = []

    print("\n=== Commencing Denoising Loop ===")
    for noisy_path in noisy_files:
        match = re.search(r"(\d+)\.h5$", noisy_path)
        if not match: continue
        sample_num = int(match.group(1))
        
        with h5py.File(noisy_path, 'r') as file:
            test_d = file['density'][:]
            
        # 1. Evaluate baseline Noisy VMC JSD
        jsd_ref = evaluate_and_enforce(test_d, ref_d_mean)
        results_ref.append((sample_num, jsd_ref))
        print(f"\n-> Sample {sample_num} loaded. Baseline JSD: {jsd_ref:.6e}")
        
        # 2. Iterate through Models
        for model_name in models_to_run:
            raw_denoised = run_model_inference(test_d, dft_d, model_name, models_dict, transform_type='residual_noise')
            jsd_score = evaluate_and_enforce(raw_denoised, ref_d_mean)
            
            results_dict[model_name].append((sample_num, jsd_score))
            print(f"   ↳ {model_name} JSD = {jsd_score:.6e}")
            
            # Save Numpy Array
            output_file = os.path.join(base_dir, f"{os.path.splitext(os.path.basename(noisy_path))[0]}_{model_name}_denoised.npy")
            np.save(output_file, raw_denoised)

    generate_log_plots(results_dict, results_ref, DFT_vs_VMC, models_to_run)

# ==========================================
# 5. UNIFIED LOG-LOG PLOTTING
# ==========================================
def generate_log_plots(results_dict, results_ref, DFT_vs_VMC, models_to_run):
    print("\n=== Generating Master Plots & Dataframes ===")
    
    # 1. Compile Dataframe
    df = pd.DataFrame(np.array(sorted(results_ref, key=lambda x: x[0])), columns=["Samples", "Noisy_VMC"])
    for model_name in models_to_run:
        model_data = np.array(sorted(results_dict[model_name], key=lambda x: x[0]))
        temp_df = pd.DataFrame(model_data, columns=["Samples", model_name])
        df = pd.merge(df, temp_df, on="Samples", how="outer")

    df = df.sort_values("Samples").reset_index(drop=True)
    df.to_csv("image_models_performance.csv", index=False)
    print("Data saved to image_models_performance.csv")

    # 2. Plotting Formatting
    matplotlib.use('Agg')
    logging.getLogger('matplotlib').setLevel(logging.WARNING)
    plt.rcParams.update({
        'font.size': 14, 'font.family': 'serif', 'axes.labelsize': 16,
        'axes.linewidth': 1.5, 'xtick.major.size': 7, 'xtick.major.width': 1.5,
        'ytick.major.size': 7, 'ytick.major.width': 1.5,
        'legend.frameon': True, 'legend.edgecolor': 'black', 'legend.fontsize': 10
    })

    fig, ax = plt.subplots(figsize=(10, 8))
    ax.plot(df["Samples"], df["Noisy_VMC"], marker="o", linestyle="--", color="red", label="Noisy VMC", alpha=0.7, markersize=8)
    
    # Isolate general image models for automated cycling colors
    filtered_models = [m for m in models_to_run if m.lower() not in ['nature', 'fft']]
    colors = plt.cm.tab10(np.linspace(0, 1, len(filtered_models)))

    # Explicitly track and plot FFT in its custom style
    ax.plot(df["Samples"], df["FFT"], marker="D", linestyle="-", color="magenta", label="FFT", linewidth=2.5, markersize=8)

    for color, model_name in zip(colors, filtered_models):
        linestyle = "--" if model_name in ["CAE", "bm3d", "CAE_3D"] else "-"
        ax.plot(df["Samples"], df[model_name], marker="s", linestyle=linestyle, label=model_name, color=color, linewidth=2.5, markersize=8)

    ax.axhline(DFT_vs_VMC, color="black", linestyle=":", label="DFT Baseline", linewidth=2)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Number of VMC Samples', fontweight='bold', labelpad=10)
    ax.set_ylabel(r'$D_{JS}$ (Jensen-Shannon Divergence)', fontweight='bold', labelpad=10)
    ax.legend(loc='best', bbox_to_anchor=(1.05, 1))
    ax.grid(False)

    plt.tight_layout()
    output_base = "image_models_denoising_convergence"
    plt.savefig(f"{output_base}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{output_base}.pdf", bbox_inches='tight')
    print(f"Publication-ready plots saved as {output_base}.png and {output_base}.pdf")

if __name__ == "__main__":
    main()

2026-05-27 14:50:12.785539: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779918612.802790  891614 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779918612.808389  891614 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779918612.823578  891614 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779918612.823594  891614 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779918612.823597  891614 computation_placer.cc:177] computation placer alr

Loading Base Data & DFT Reference...
Found 33 noisy files.
Pre-loading Models into Memory...
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block Initial Type: SW, drop_path_rate:0.000000
Block Initial Type: W, drop_path_rate:0.000000
Block

I0000 00:00:1779918759.838865  891614 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38479 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:c1:00.0, compute capability: 8.0



=== Commencing Denoising Loop ===

-> Sample 10240 loaded. Baseline JSD: 4.474564e-01
   ↳ scunet_trained JSD = 4.240757e-02
   ↳ scunet_pre JSD = 8.500095e-02
   ↳ scunet_ft JSD = 2.736160e-02


I0000 00:00:1779918777.177918  892149 service.cc:152] XLA service 0x7ff0280245c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779918777.177945  892149 service.cc:160]   StreamExecutor device (0): NVIDIA A100-SXM4-40GB, Compute Capability 8.0
2026-05-27 14:52:57.204211: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1779918777.447754  892149 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1779918778.555612  892149 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


   ↳ nature JSD = 1.447346e-02
   ↳ CAE JSD = 7.970118e-02
   ↳ CAE_3D JSD = 1.569811e-04
   ↳ bm3d JSD = 3.657047e-03
   ↳ FFT JSD = 1.633966e-04

-> Sample 20480 loaded. Baseline JSD: 2.926950e-01
   ↳ scunet_trained JSD = 2.891531e-02
   ↳ scunet_pre JSD = 9.654257e-02
   ↳ scunet_ft JSD = 2.786439e-02
   ↳ nature JSD = 1.432576e-02
   ↳ CAE JSD = 6.664140e-02
   ↳ CAE_3D JSD = 1.550171e-04
   ↳ bm3d JSD = 1.761575e-03
   ↳ FFT JSD = 7.577015e-05

-> Sample 40960 loaded. Baseline JSD: 1.693626e-01
   ↳ scunet_trained JSD = 2.411294e-02
   ↳ scunet_pre JSD = 6.586880e-02
   ↳ scunet_ft JSD = 2.416721e-02
   ↳ nature JSD = 1.260000e-02
   ↳ CAE JSD = 2.718985e-02
   ↳ CAE_3D JSD = 1.522255e-04
   ↳ bm3d JSD = 9.353179e-04
   ↳ FFT JSD = 7.771239e-05

-> Sample 81920 loaded. Baseline JSD: 9.013380e-02
   ↳ scunet_trained JSD = 2.083153e-02
   ↳ scunet_pre JSD = 2.617244e-02
   ↳ scunet_ft JSD = 1.914806e-02
   ↳ nature JSD = 1.732370e-02
   ↳ CAE JSD = 2.343238e-02
   ↳ CAE_3D JSD = 1.

[DEBUG] Loaded backend Agg version v2.2.


Publication-ready plots saved as image_models_denoising_convergence.png and image_models_denoising_convergence.pdf


In [3]:
#!/usr/bin/env python3
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import logging

# ==========================================
# CONFIGURATION
# ==========================================
CSV_PATH = "image_models_performance.csv"
OUTPUT_BASE = "image_models_denoising_convergence"

# IMPORTANT: Replace this with your actual DFT vs VMC JSD value 
# from your previous terminal outputs.
DFT_BASELINE_VALUE = 0.001  

def main():
    print(f"Loading data from {CSV_PATH}...")
    try:
        df = pd.read_csv(CSV_PATH)
    except FileNotFoundError:
        print(f"Error: Could not find {CSV_PATH}. Ensure you are in the correct directory.")
        return

    # Plotting Formatting
    matplotlib.use('Agg')
    logging.getLogger('matplotlib').setLevel(logging.WARNING)
    plt.rcParams.update({
        'font.size': 14, 'font.family': 'serif', 'axes.labelsize': 16,
        'axes.linewidth': 1.5, 'xtick.major.size': 8, 'xtick.major.width': 1.5,
        'ytick.major.size': 8, 'ytick.major.width': 1.5,
        'legend.frameon': True, 'legend.edgecolor': 'black', 'legend.fontsize': 11
    })

    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Plot Noisy VMC Reference
    if "Noisy_VMC" in df.columns:
        ax.plot(df["Samples"], df["Noisy_VMC"], marker="o", linestyle="--", color="red", label="Noisy VMC", alpha=0.7, markersize=8)
    
    # Identify all models in the CSV excluding standard columns
    all_models = [col for col in df.columns if col not in ["Samples", "Noisy_VMC"]]
    
    # Isolate general image models for automated cycling colors (EXCLUDING CAE_3D, nature, and FFT)
    filtered_models = [m for m in all_models if m.lower() not in ['nature', 'fft', 'cae_3d']]
    colors = plt.cm.tab10(np.linspace(0, 1, len(filtered_models)))
    
    # List of distinct markers to make each line clearly identifiable
    distinct_markers = ['s', '^', 'v', 'p', '*', 'X', 'h']

    # Explicitly track and plot FFT in its custom style if it exists
    if "FFT" in df.columns:
        ax.plot(df["Samples"], df["FFT"], marker="D", linestyle="-", color="magenta", label="FFT", linewidth=2.5, markersize=8)

    # Plot the filtered image models with unique markers
    for i, (color, model_name) in enumerate(zip(colors, filtered_models)):
        linestyle = "--" if model_name in ["CAE", "bm3d"] else "-"
        marker_style = distinct_markers[i % len(distinct_markers)]
        
        ax.plot(df["Samples"], df[model_name], marker=marker_style, linestyle=linestyle, 
                label=model_name, color=color, linewidth=2.5, markersize=8)

    # Plot DFT Baseline
    ax.axhline(DFT_BASELINE_VALUE, color="black", linestyle=":", label="DFT Baseline", linewidth=2)

    # Axis Formatting
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Number of VMC Samples', fontweight='bold', labelpad=10)
    ax.set_ylabel(r'$D_{JS}$ (Jensen-Shannon Divergence)', fontweight='bold', labelpad=10)
    
    # Add minor ticks for logarithmic scales for better readability
    ax.tick_params(axis='both', which='minor', direction='in', length=4, width=1)
    ax.tick_params(axis='both', which='major', direction='in', length=8, width=1.5)

    # Legend inside the plot area
    ax.legend(loc='best')
    ax.grid(False)

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_BASE}.png", dpi=300, bbox_inches='tight')
    plt.savefig(f"{OUTPUT_BASE}.pdf", bbox_inches='tight')
    
    print(f"Publication-ready plots saved as {OUTPUT_BASE}.png and {OUTPUT_BASE}.pdf")

if __name__ == "__main__":
    main()

Loading data from image_models_performance.csv...
Publication-ready plots saved as image_models_denoising_convergence.png and image_models_denoising_convergence.pdf
